In [34]:
import duckdb
import pandas as pd
import sys, pathlib

sys.path.insert(0, str(pathlib.Path('../src').resolve()))
import irp.config as _config

cfg  = _config.load()
_ROOT = pathlib.Path(_config.__file__).parents[2]
DB   = str((_ROOT / cfg['store']['db_path']).resolve())

def q(sql, params=None):
    with duckdb.connect(DB, read_only=True) as con:
        return con.execute(sql, params or []).df()

print('DB:', DB)

DB: /mnt/Dev/active_python_projects/investment_research_platform/data/irp.duckdb


In [26]:
df = q('SELECT * FROM prices')
print(df.shape)
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(45548912, 10)


,ticker,source_id,source,date,open,high,low,close,volume,inserted_at
0,5YUSY,5yusy.b,stooq,1999-09-07,5.822,5.822,5.822,5.822,0.0,2026-04-29 22:51:01.500638+02:00
1,5YUSY,5yusy.b,stooq,1999-11-17,5.977,5.977,5.977,5.977,0.0,2026-04-29 22:51:01.500638+02:00
2,5YUSY,5yusy.b,stooq,1999-12-03,6.074,6.074,6.074,6.074,0.0,2026-04-29 22:51:01.500638+02:00
3,5YUSY,5yusy.b,stooq,2000-04-05,6.133,6.133,6.133,6.133,0.0,2026-04-29 22:51:01.500638+02:00
4,5YUSY,5yusy.b,stooq,2000-04-07,6.174,6.174,6.174,6.174,0.0,2026-04-29 22:51:01.500638+02:00


In [35]:
mask =df['source_id'].str[-2:] == '.m'
print(mask.sum())
df[mask]

0


,ticker,source_id,source,date,open,high,low,close,volume,inserted_at


In [13]:
df['source_id'].str[-2:]

0           .b
1           .b
2           .b
3           .b
4           .b
            ..
45527581    us
45527582    us
45527583    us
45527584    us
45527585    us
Name: source_id, Length: 45527586, dtype: str

In [ ]:
df.dtypes

In [ ]:
# single ticker
tic = 'aac.us'
q(f"SELECT * FROM prices WHERE source_id = '{tic}' ORDER BY date")


,ticker,source_id,source,date,open,high,low,close,volume,inserted_at


In [41]:
# single ticker
tic = 'SYNA'
q(f"SELECT * FROM prices WHERE ticker = '{tic}' ORDER BY date")


,ticker,source_id,source,date,open,high,low,close,volume,inserted_at
0,SYNA,syna.us,stooq,2005-02-25,16.50,16.8900,15.870,16.22,26373825.0,2026-04-29 22:55:11.848921+02:00
1,SYNA,syna.us,stooq,2005-02-28,16.63,16.6300,15.850,15.93,3330367.0,2026-04-29 22:55:11.848921+02:00
2,SYNA,syna.us,stooq,2005-03-01,16.00,16.1500,15.370,15.51,2064933.0,2026-04-29 22:55:11.848921+02:00
3,SYNA,syna.us,stooq,2005-03-02,15.49,15.8300,15.170,15.37,1694953.0,2026-04-29 22:55:11.848921+02:00
4,SYNA,syna.us,stooq,2005-03-03,15.38,15.5300,14.990,15.05,1764157.0,2026-04-29 22:55:11.848921+02:00
...,...,...,...,...,...,...,...,...,...,...
5321,SYNA,syna.us,stooq,2026-04-23,85.91,87.6600,83.050,84.95,472014.0,2026-04-30 00:13:47.397391+02:00
5322,SYNA,syna.us,stooq,2026-04-24,87.14,94.2200,86.645,93.86,1815769.0,2026-04-30 00:13:47.397391+02:00
5323,SYNA,syna.us,stooq,2026-04-27,92.90,93.7599,90.300,91.73,936389.0,2026-04-30 00:13:47.397391+02:00
5324,SYNA,syna.us,stooq,2026-04-28,89.15,90.1750,85.520,86.20,599794.0,2026-04-30 00:13:47.397391+02:00


In [28]:
coverage = q("""
    SELECT ticker, source_id, MIN(date) AS first_date, MAX(date) AS last_date, COUNT(*) AS n_rows
    FROM prices
    GROUP BY ticker, source_id
    ORDER BY ticker
""")
coverage

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ticker,source_id,first_date,last_date,n_rows
0,10YATY,10yaty.b,2006-04-24,2026-04-16,5120
1,10YAUY,10yauy.b,2005-11-28,2026-04-16,5195
2,10YBEY,10ybey.b,1993-12-06,2026-04-16,8282
3,10YBRY,10ybry.b,2010-06-04,2026-04-16,3963
4,10YCAP,10ycap.b,2024-03-21,2026-04-16,514
...,...,...,...,...,...
14439,^_UK,^_uk,2016-02-16,2026-04-15,2575
14440,^_US,^_us,2016-02-16,2026-04-16,2558
14441,^_USNM,^_usnm,2016-02-16,2026-04-16,2557
14442,^_USNQ,^_usnq,2016-02-16,2026-04-16,2558


In [ ]:
coverage.sort_values('first_date').head(10)


In [17]:
# Delete MSFT quotes from 2026-04-17 onwards (to test upsert re-insert)
with duckdb.connect(DB) as con:
    deleted = con.execute(
        "DELETE FROM prices WHERE date >= '2026-04-16'"
    ).rowcount
print(f"Deleted {deleted} rows")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Deleted -1 rows


In [ ]:
mask = coverage['last_date'] >= '2026-04-24'
coverage[mask].sort_values('last_date')


In [ ]:
coverage[coverage['n_rows']<=5]
